# Strategy Scanner
Runs buy/sell zone + trend confluence across a watchlist and shows ranked results.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
import warnings; warnings.filterwarnings('ignore')

from data_fetch.data_fetch import fetch_batch, get_watchlist
from scanner import scan_watchlist
print('✅ Ready')

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
WATCHLIST       = 'nifty50'   # 'nifty50' | 'next50' | 'midcap' | 'all_nifty'
INTERVAL        = '1d'
DAYS            = 500
ORDER           = 5
DIRECTION       = 'buy'       # 'buy' | 'sell'
MIN_CONFLUENCE  = 0.45
MAX_WORKERS     = 8

In [ ]:
# ── Fetch data ────────────────────────────────────────────────────────────────
tickers = get_watchlist(WATCHLIST)
print(f'Fetching {len(tickers)} tickers ({INTERVAL})...')
data_dict   = fetch_batch(tickers, interval=INTERVAL,  days=DAYS, max_workers=MAX_WORKERS)
weekly_dict = fetch_batch(tickers, interval='1wk',     days=DAYS*2, max_workers=MAX_WORKERS)
print(f'✅ Got data for {len(data_dict)} tickers')

In [ ]:
# ── Run scanner ───────────────────────────────────────────────────────────────
results = scan_watchlist(
    tickers=list(data_dict.keys()),
    data_dict=data_dict,
    weekly_dict=weekly_dict,
    order=ORDER,
    direction=DIRECTION,
    min_confluence=MIN_CONFLUENCE,
    max_workers=MAX_WORKERS,
)
print(f'✅ {len(results)} setups found')
results

In [ ]:
# ── Visual summary ────────────────────────────────────────────────────────────
if not results.empty:
    top = results.head(15)
    fig = go.Figure(go.Bar(
        x=top['ticker'].str.replace('.NS',''),
        y=top['confluence'],
        text=top['trend'],
        textposition='auto',
        marker_color=['#22c55e' if s > 0.65 else '#3b82f6' if s > 0.5 else '#f59e0b'
                      for s in top['confluence']],
    ))
    fig.update_layout(
        title='Top Setups by Confluence Score',
        yaxis_title='Confluence Score',
        template='plotly_dark', height=400
    )
    fig.show()

In [ ]:
# ── Inspect top setup ─────────────────────────────────────────────────────────
if not results.empty:
    INSPECT = results.iloc[0]['ticker']
    from algos.pivot_engine import extrems, detect_trend
    from algos.zones import buy_zone, sell_zone
    import plotly.graph_objects as go

    df = data_dict[INSPECT]
    pivots = extrems(df, order=ORDER)
    trend  = detect_trend(pivots)
    zones, _ = buy_zone(df, 0, 'NA') if DIRECTION=='buy' else (sell_zone(df,0), 0)

    fig = go.Figure(go.Candlestick(
        x=df.index, open=df['Open'], high=df['High'],
        low=df['Low'], close=df['Close']
    ))
    pivot_dates = [df.index[i] for i,_,_ in pivots]
    pivot_vals  = [v for _,v,_ in pivots]
    pivot_cols  = ['red' if t=='T' else 'lime' for _,_,t in pivots]
    fig.add_trace(go.Scatter(
        x=pivot_dates, y=pivot_vals, mode='markers+lines',
        marker=dict(color=pivot_cols, size=8),
        line=dict(color='rgba(167,139,250,0.5)', width=1),
        name='Pivots'
    ))
    for z in zones:
        fig.add_vrect(x0=z[0], x1=z[1], fillcolor='rgba(34,197,94,0.12)',
                      opacity=0.8, layer='below', line_width=0)
    fig.update_layout(
        title=f'{INSPECT} · {trend}', template='plotly_dark',
        xaxis_rangeslider_visible=False, height=600
    )
    fig.show()